In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import MBartForConditionalGeneration, MBartConfig, PreTrainedModel, MBart50TokenizerFast
from transformers import AdamW, get_linear_schedule_with_warmup
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from nltk.translate.bleu_score import corpus_bleu
import nltk
nltk.download('punkt')

# Set the CUDA_VISIBLE_DEVICES environment variable
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Check if CUDA is available and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Print GPU information if available
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

def preprocess_verb_data(verbs_data):
    processed_data = []
    for item in verbs_data:
        sentence = item['sentence']
        translation = item['translation']
        verb_infos = [item['verb_info']] if isinstance(item['verb_info'], dict) else item['verb_info']
        
        processed_sentence = sentence
        for verb_info in verb_infos:
            verb_token = f"<VERB:{verb_info['lemma']}:" \
                         f"<TENSE:{verb_info['analysis'].get('tense', 'UNKNOWN')}>:" \
                         f"<ASPECT:{verb_info['analysis'].get('aspect', 'UNKNOWN')}>:" \
                         f"<MOOD:{verb_info['analysis'].get('mood', 'UNKNOWN')}>:" \
                         f"<VOICE:{verb_info['analysis'].get('voice', 'UNKNOWN')}>:" \
                         f"<PERSON:{verb_info['analysis'].get('person', 'UNKNOWN')}>:" \
                         f"<NUMBER:{verb_info['analysis'].get('number', 'UNKNOWN')}>"
            
            processed_sentence = processed_sentence.replace(verb_info['word'], verb_token)
        
        processed_data.append({
            'original_sentence': sentence,
            'processed_sentence': processed_sentence,
            'translation': translation,
            'verb_infos': verb_infos
        })
    
    return processed_data

def split_data(processed_data, test_size=0.1, val_size=0.1):
    train_val, test = train_test_split(processed_data, test_size=test_size, random_state=42)
    train, val = train_test_split(train_val, test_size=val_size/(1-test_size), random_state=42)
    return train, val, test

class ArmenianVerbGenerator(PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.mbart = MBartForConditionalGeneration.from_pretrained('facebook/mbart-large-50')
        self.config = self.mbart.config
        
        self.morpho_classifiers = nn.ModuleDict({
            'tense': nn.Linear(self.config.d_model, 15),
            'aspect': nn.Linear(self.config.d_model, 4),
            'mood': nn.Linear(self.config.d_model, 14),
            'voice': nn.Linear(self.config.d_model, 5),
            'person': nn.Linear(self.config.d_model, 4),
            'number': nn.Linear(self.config.d_model, 4)
        })
        
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask, labels=None, **morpho_labels):
        outputs = self.mbart(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            output_hidden_states=True
        )
    
        hidden_states = outputs.encoder_last_hidden_state[:, 0, :]  # Use [CLS] token
    
        morpho_logits = {feature: self.morpho_classifiers[feature](hidden_states) for feature in self.morpho_classifiers}
    
        loss = outputs.loss
        if morpho_labels:
            morpho_loss = sum(F.cross_entropy(morpho_logits[feature], morpho_labels[f"{feature}_labels"]) 
                              for feature in self.morpho_classifiers.keys()
                              if f"{feature}_labels" in morpho_labels)
            loss += 0.5 * morpho_loss

        return loss, outputs.logits, morpho_logits

    def generate(self, input_ids, attention_mask=None, **kwargs):
        return self.mbart.generate(input_ids, attention_mask=attention_mask, **kwargs)

class ArmenianVerbDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=128):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.morpho_features = ['tense', 'aspect', 'mood', 'voice', 'person', 'number']
        self.feature_to_id = {
            'tense': {'past': 0, 'present': 1, 'future': 2, 'present perfect': 3, 'pluperfect': 4, 
                      'imperfect': 5, 'aorist': 6, 'infinitive': 7, 'future perfect': 8, 'present progressive': 9, 
                      'future progressive': 10, 'future participle': 11, 'future in the past': 12, 
                      'present participle': 13, 'future infinitive': 14},
            'aspect': {'perfective': 0, 'imperfective': 1, 'habitual': 2, 'inceptive': 3},
            'mood': {'indicative': 0, 'subjunctive': 1, 'imperative': 2, 'conditional': 3, 'optative': 4, 
                     'necessitative': 5, 'presumptive': 6, 'interrogative': 7, 'infinitive': 8, 
                     'irrealis': 9, 'potential': 10, 'hortative': 11, 'obligative': 12, 'UNKNOWN': 13},
            'voice': {'active': 0, 'passive': 1, 'middle': 2, 'reflexive': 3, 'causative': 4},
            'person': {'1': 0, '2': 1, '3': 2, 'UNKNOWN': 3},
            'number': {'singular': 0, 'plural': 1, 'UNKNOWN': 2, 'invariable': 3}
        }
    
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        input_text = item['processed_sentence']
        target_text = item['original_sentence']
        
        inputs = self.tokenizer(input_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        targets = self.tokenizer(target_text, max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        
        input_ids = inputs.input_ids.squeeze()
        attention_mask = inputs.attention_mask.squeeze()
        labels = targets.input_ids.squeeze()
        
        morpho_labels = {}
        for feature in self.morpho_features:
            value = item['verb_infos'][0]['analysis'].get(feature, 'UNKNOWN')
            morpho_labels[f"{feature}_labels"] = torch.tensor(self.feature_to_id[feature].get(value, len(self.feature_to_id[feature]) - 1))
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            **morpho_labels
        }

def calculate_bleu(references, hypotheses):
    return corpus_bleu([[r.split()] for r in references], [h.split() for h in hypotheses])



In [ ]:
from sklearn.metrics import accuracy_score


def evaluate(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    all_references = []
    all_hypotheses = []
    morpho_preds = {feature: [] for feature in model.morpho_classifiers.keys()}
    morpho_labels = {feature: [] for feature in model.morpho_classifiers.keys()}
    
    with torch.no_grad():
        for batch in dataloader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            
            loss, _, morpho_logits = model(**inputs)
            
            total_loss += loss.item()
            
            # Generate translations
            generated = model.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'])
            
            hypotheses = tokenizer.batch_decode(generated, skip_special_tokens=True)
            references = tokenizer.batch_decode(inputs['labels'], skip_special_tokens=True)
            
            all_hypotheses.extend(hypotheses)
            all_references.extend(references)
            
            # Collect morphological predictions and labels
            for feature in morpho_logits.keys():
                morpho_preds[feature].extend(morpho_logits[feature].argmax(dim=-1).cpu().tolist())
                morpho_labels[feature].extend(inputs[f"{feature}_labels"].cpu().tolist())
    
    avg_loss = total_loss / len(dataloader)
    bleu_score = calculate_bleu(all_references, all_hypotheses)
    
    morpho_accuracies = {}
    for feature in morpho_preds.keys():
        morpho_accuracies[feature] = accuracy_score(morpho_labels[feature], morpho_preds[feature])
    
    return avg_loss, bleu_score, morpho_accuracies


def train(model, train_dataloader, val_dataloader, optimizer, scheduler, tokenizer, device, num_epochs=10, save_dir='./model_checkpoints'):
    model.to(device)
    
    best_val_loss = float('inf')
    training_history = []
    
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            optimizer.zero_grad()
            
            inputs = {k: v.to(device) for k, v in batch.items()}
            
            loss, _, _ = model(**inputs)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            total_loss += loss.item()
        
        avg_train_loss = total_loss / len(train_dataloader)
        
        # Validation
        val_loss, bleu_score, morpho_accuracies = evaluate(model, val_dataloader, tokenizer, device)
        
        epoch_info = {
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_loss': val_loss,
            'bleu_score': bleu_score,
            'morpho_accuracies': morpho_accuracies
        }
        training_history.append(epoch_info)
        
        print(f"Epoch {epoch+1}")
        print(f"  Training Loss: {avg_train_loss:.4f}")
        print(f"  Validation Loss: {val_loss:.4f}")
        print(f"  Validation BLEU Score: {bleu_score:.4f}")
        print("  Morphological Accuracies:")
        for feature, accuracy in morpho_accuracies.items():
            print(f"    {feature}: {accuracy:.4f}")
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            model_path = os.path.join(save_dir, f'best_model_epoch_{epoch+1}')
            model.save_pretrained(model_path)
            print(f"  Best model saved to {model_path}")
        
        if (epoch + 1) % 5 == 0:
            model_path = os.path.join(save_dir, f'model_epoch_{epoch+1}')
            model.save_pretrained(model_path)
            print(f"  Model checkpoint saved to {model_path}")

    return model, training_history

if __name__ == "__main__":
    from armenian_verbs import verbs_data

    # Preprocess and split the data
    processed_data = preprocess_verb_data(verbs_data)
    train_data, val_data, test_data = split_data(processed_data)

    # Initialize tokenizer and create datasets
    tokenizer = MBart50TokenizerFast.from_pretrained('facebook/mbart-large-50')
    train_dataset = ArmenianVerbDataset(train_data, tokenizer)
    val_dataset = ArmenianVerbDataset(val_data, tokenizer)
    test_dataset = ArmenianVerbDataset(test_data, tokenizer)

    # Create dataloaders
    train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=16)
    test_dataloader = DataLoader(test_dataset, batch_size=16)

    # Initialize model, optimizer, and scheduler
    config = MBartConfig.from_pretrained('facebook/mbart-large-50')
    model = ArmenianVerbGenerator(config)
    optimizer = AdamW(model.parameters(), lr=5e-5)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=500, num_training_steps=len(train_dataloader) * 10)

    # Train the model
    save_dir = './model_checkpoints'
    os.makedirs(save_dir, exist_ok=True)
    trained_model, training_history = train(model, train_dataloader, val_dataloader, optimizer, scheduler, tokenizer, device, save_dir=save_dir)

    # Save the final model
    final_model_path = os.path.join(save_dir, 'final_model')
    trained_model.save_pretrained(final_model_path)
    print(f"Final model saved to {final_model_path}")

    # Analyze training history
    print("\n--- Training History Analysis ---\n")
    for epoch_info in training_history:
        print(f"Epoch {epoch_info['epoch']}:")
        print(f"  Training Loss: {epoch_info['train_loss']:.4f}")
        print(f"  Validation Loss: {epoch_info['val_loss']:.4f}")
        print(f"  Validation BLEU Score: {epoch_info['bleu_score']:.4f}")
        print("  Morphological Accuracies:")
        for feature, accuracy in epoch_info['morpho_accuracies'].items():
            print(f"    {feature}: {accuracy:.4f}")
        print()

    # You can add more analysis here, such as plotting learning curves
    # For example, using matplotlib:
    import matplotlib.pyplot as plt

    epochs = [info['epoch'] for info in training_history]
    train_losses = [info['train_loss'] for info in training_history]
    val_losses = [info['val_loss'] for info in training_history]
    bleu_scores = [info['bleu_score'] for info in training_history]

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Train Loss')
    plt.plot(epochs, val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, bleu_scores)
    plt.xlabel('Epoch')
    plt.ylabel('BLEU Score')

    plt.tight_layout()
    plt.savefig('training_history.png')
    print("Training history plot saved as 'training_history.png'")

In [ ]:
def evaluate_model(model, dataloader, tokenizer, device):
    model.eval()
    total_loss = 0
    all_translations = []
    all_references = []
    all_morpho_preds = {feature: [] for feature in model.morpho_classifiers.keys()}
    all_morpho_true = {feature: [] for feature in model.morpho_classifiers.keys()}

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            inputs = {k: v.to(device) for k, v in batch.items()}
            
            loss, _, morpho_logits = model(**inputs)
            total_loss += loss.item()
            
            generated = model.generate(inputs['input_ids'], attention_mask=inputs['attention_mask'])
            
            translations = tokenizer.batch_decode(generated, skip_special_tokens=True)
            references = tokenizer.batch_decode(inputs['labels'], skip_special_tokens=True)
            
            all_translations.extend(translations)
            all_references.extend(references)
            
            for feature in morpho_logits.keys():
                preds = morpho_logits[feature].argmax(dim=-1).cpu().numpy()
                true = inputs[f"{feature}_labels"].cpu().numpy()
                all_morpho_preds[feature].extend(preds)
                all_morpho_true[feature].extend(true)

    avg_loss = total_loss / len(dataloader)
    bleu_score = corpus_bleu([[ref.split()] for ref in all_references], [hyp.split() for hyp in all_translations])
    
    morpho_accuracies = {}
    for feature in all_morpho_preds.keys():
        morpho_accuracies[feature] = accuracy_score(all_morpho_true[feature], all_morpho_preds[feature])

    return avg_loss, bleu_score, morpho_accuracies, all_translations, all_references, all_morpho_preds, all_morpho_true

# Evaluate on test data
print("\n--- Evaluating Enhanced Model on Test Data ---\n")
test_loss, test_bleu, test_morpho_accuracies, test_translations, test_references, test_morpho_preds, test_morpho_true = evaluate_model(trained_model, test_dataloader, tokenizer, device)

print(f"Enhanced Model Test Loss: {test_loss:.4f}")
print(f"Enhanced Model Test BLEU Score: {test_bleu:.4f}")
print("Enhanced Model Test Morphological Accuracies:")
for feature, accuracy in test_morpho_accuracies.items():
    print(f"  {feature.capitalize()}: {accuracy:.4f}")

# Evaluate on inference data
print("\n--- Evaluating Enhanced Model on Inference Data ---\n")

# Preprocess inference data
processed_inference_data = preprocess_verb_data(inference_verbs_data)

# Create inference dataset and dataloader
inference_dataset = ArmenianVerbDataset(processed_inference_data, tokenizer)
inference_dataloader = DataLoader(inference_dataset, batch_size=16, shuffle=False)

inf_loss, inf_bleu, inf_morpho_accuracies, inf_translations, inf_references, inf_morpho_preds, inf_morpho_true = evaluate_model(trained_model, inference_dataloader, tokenizer, device)

print(f"Enhanced Model Inference Loss: {inf_loss:.4f}")
print(f"Enhanced Model Inference BLEU Score: {inf_bleu:.4f}")
print("Enhanced Model Inference Morphological Accuracies:")
for feature, accuracy in inf_morpho_accuracies.items():
    print(f"  {feature.capitalize()}: {accuracy:.4f}")

# Print detailed results for inference data
print("\nDetailed Inference Results:")
for i in range(len(inf_translations)):
    print(f"\nInstance {i+1}:")
    print(f"Input: {tokenizer.decode(inference_dataset[i]['input_ids'], skip_special_tokens=True)}")
    print(f"Reference: {inf_references[i]}")
    print(f"Model Output: {inf_translations[i]}")
    print("Morphological features:")
    for feature in inf_morpho_preds.keys():
        print(f"  {feature.capitalize()}: True - {inf_morpho_true[feature][i]}, Predicted - {inf_morpho_preds[feature][i]}")
    print("-" * 50)